## 스펙트로그램과 특징추출 실습
본 페이지에서는 음성파일의 스펙트로그램과 멜스펙트로그램을 그려보는 실습을 진행합니다.

## 환경 설정

실습은 a1004/local 디렉토리에서 수행합니다. 이 디렉토리로 이동합니다.

# 1. venv 활성화
source ~/espnet/tools/activate_python.sh

# 2. jupyter 설치 (최초 1회)
pip install jupyter

# 3. 실행
jupyter notebook

### 로그멜 스펙트로그램

먼저 음성파일을 들어봅니다.

In [ ]:
from IPython.display import Audio
filename = "data/KsponSpeech_E00001.wav"
Audio(filename, autoplay=False)

librosa 패키지는 음성파일을 다루는데 널리 쓰이는 파이선 모듈입니다.
이 모듈을 이용해서 파일을 로드하고 스펙트로그램을 그립니다.
먼저 librosa 모듈을 설치합니다.

In [ ]:
!pip install librosa

In [ ]:
import librosa
samples, sampling_rate = librosa.load(filename, sr=None)
len(samples), sampling_rate

In [ ]:
!pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.figsize"]=12,5
from librosa import display
import numpy as np
plt.figure()
librosa.display.waveshow(y = samples, sr = sampling_rate, color='blue')
plt.show()

먼저 스펙트로그램을 그려봅니다. 변수들을 바꾸어가면서 어떤 변화가 있는지 관찰해봅니다.

In [ ]:
hop_length = 160        # 프레임 이동 간격 (샘플 수), 10ms @ 16kHz
n_fft = 512             # FFT 크기, 주파수 해상도 결정
win_length = 320        # 윈도우 크기 (샘플 수), 20ms @ 16kHz
window = 'hann'         # 윈도우 함수 종류

stft = librosa.stft(samples, n_fft=n_fft, win_length=win_length, hop_length=hop_length)  # 단시간 푸리에 변환 (shape: n_fft/2+1 x frames)
spectrogram = np.abs(stft)**2   # 복소수 → 파워 스펙트로그램
log_spectrogram = librosa.power_to_db(spectrogram, ref=np.max)  # 파워를 dB 스케일로 변환

plt.figure()
librosa.display.specshow(log_spectrogram, sr=sampling_rate, hop_length=hop_length, y_axis="hz", x_axis="time")  # y축: 주파수(Hz), x축: 시간(초)


이제 멜스펙트로그램을 그려봅니다. 변수들을 바꾸어가면서 어떤 변화가 있는지 관찰해봅니다.

In [ ]:
hop_length = 160        # 프레임 이동 간격 (샘플 수), 10ms @ 16kHz
n_fft = 512             # FFT 크기, 주파수 해상도 결정
win_length = 320        # 윈도우 크기 (샘플 수), 20ms @ 16kHz
window = 'hann'         # 윈도우 함수 종류

n_mels = 40             # 멜 필터뱅크 개수 (특징 벡터 차원)
fmin = 0                # 멜 필터 최저 주파수 (Hz)
fmax = None             # 멜 필터 최고 주파수 (None이면 sr/2)

S = librosa.feature.melspectrogram(
    y=samples,
    sr=sampling_rate,
    hop_length=hop_length, n_fft=n_fft, win_length=win_length, window=window,
    n_mels=n_mels, fmin=fmin, fmax=fmax)   # 멜스펙트로그램 계산 (shape: n_mels x frames)
S_dB = librosa.power_to_db(S, ref=np.max)  # 파워를 dB 스케일로 변환
dim_feature, len_feature = S_dB.shape      # dim_feature=멜차원, len_feature=프레임수
fig = plt.figure()
librosa.display.specshow(S_dB, y_axis=None, x_axis=None)
fig.gca().set_yticks(range(0, dim_feature+1, 10))   # y축 눈금: 멜 인덱스
fig.gca().set_xticks(range(0, len_feature, 100))     # x축 눈금: 프레임 인덱스
fig.gca().set_ylabel("Mel-freq. Index")
fig.gca().set_xlabel("Frame Index")


### SpecAug

구글에서 발표한 SpecAug 알고리즘에 대해서 실습해봅니다.
SpecAug 모듈을 초기화합니다. 각 파라미터의 값을 변경하면서 시도해 봅니다.

In [ ]:
from espnet2.asr.specaug.specaug import SpecAug
import torch

specaug = SpecAug(apply_time_warp=True,              # 시간축 워핑 적용 여부
                  time_warp_window=5,                # 시간축 워핑 범위 (프레임 수)
                  num_freq_mask=2,                   # 주파수 마스크 개수
                  apply_freq_mask=True,              # 주파수 마스크 적용 여부
                  freq_mask_width_range=(0, 30),     # 주파수 마스크 폭 범위 (멜 bin 수)
                  apply_time_mask=True,              # 시간 마스크 적용 여부
                  num_time_mask=2,                   # 시간 마스크 개수
                  time_mask_width_range=(0, 40),     # 시간 마스크 폭 범위 (프레임 수)
                 )
print(specaug)

실제 SpecAug를 적용합니다. 여러번 하면서 변화를 살펴봅니다.

In [ ]:
S_dB_in = torch.unsqueeze(torch.from_numpy(S_dB.copy().T), 0)  # numpy → tensor 변환, (1, frames, n_mels) 형태로 변환
specaug(S_dB_in)                                                 # SpecAug 적용 (in-place)
S_dB_out = torch.squeeze(S_dB_in, 0).numpy().T                  # (frames, n_mels) → numpy 변환 후 (n_mels, frames)로 복원

plt.figure()
librosa.display.specshow(S_dB_out, sr=sampling_rate, hop_length=hop_length, y_axis="mel", x_axis="time")  # y축: 멜 주파수, x축: 시간(초)